<a href="https://colab.research.google.com/github/Bindiya-Anand/IndiaKart_E-Commerce_Analytics/blob/main/Notebooks/Phase3_KPI_Bindiya_Anand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Internmo Data Analytics Project
## IndiaKart - E-Com
### Data Analyst Intern: Bindiya Anand
#### Intern Id: Inter563


# Phase 3: KPI Calculations & Identification

## Loading Datasets

In [2]:
# Loading datasets
import pandas as pd

DATE_FORMAT = "%d-%m-%Y"

orders = pd.read_csv('/content/orders DA.csv')
order_items = pd.read_csv('/content/order_items DA.csv')
customers   = pd.read_csv('/content/customers DA.csv')
payments    = pd.read_csv('/content/payments DA.csv')
returns     = pd.read_csv('/content/returns DA.csv')
inventory   = pd.read_csv('/content/inventory DA.csv')

print("orders     :", len(orders))
print("order_items:", len(order_items))
print("customers  :", len(customers))
print("payments   :", len(payments))
print("returns    :", len(returns))
print("inventory  :", len(inventory))

orders     : 50000
order_items: 100000
customers  : 10000
payments   : 50000
returns    : 10000
inventory  : 1000


## Basic Validation of Required Columns

In [11]:
required_columns = {
    "orders":      ["order_id", "order_date", "status", "final_amount"],
    "order_items": ["order_id", "category", "total_price"],
    "customers":   ["customer_id", "segment", "total_spent"],
    "returns":     ["order_id"],
}
tables = {"orders": orders, "order_items": order_items, "customers": customers, "returns": returns}

for table_name in required_columns:
    missing = []
    for col in required_columns[table_name]:
        if col not in tables[table_name].columns:
            missing.append(col)
    if len(missing) == 0:
        print(table_name, ": OK")
    else:
        print(table_name, ": MISSING ->", missing)

# Look at the real column names
print()
print("payments columns :", list(payments.columns))
print("inventory columns:", list(inventory.columns))

orders : OK
order_items : OK
customers : OK
returns : OK

payments columns : ['payment_id', 'order_id', 'customer_id', 'payment_date', 'payment_time', 'payment_method', 'amount', 'status', 'transaction_id', 'bank_name', 'gateway', 'refund_amount', 'refund_date']
inventory columns: ['inventory_id', 'product_id', 'warehouse_location', 'quantity_available', 'quantity_reserved', 'reorder_level', 'reorder_quantity', 'last_restocked_date', 'unit_cost', 'total_inventory_value', 'status']


In [8]:
PAYMENT_STATUS_COL   = 'status'    # column holding Success / Failed
INVENTORY_STATUS_COL = 'status'   # column holding In Stock / Out of Stock

# Confirm with value_counts below.
PAYMENT_FAILED_VALUE = "Failed"
INVENTORY_IN_STOCK_VALUE = "In Stock"

if PAYMENT_STATUS_COL not in payments.columns:
    raise ValueError("Set PAYMENT_STATUS_COL to a real column name from the payments list above.")
if INVENTORY_STATUS_COL not in inventory.columns:
    raise ValueError("Set INVENTORY_STATUS_COL to a real column name from the inventory list above.")

print(payments[PAYMENT_STATUS_COL].value_counts())
print()
print(inventory[INVENTORY_STATUS_COL].value_counts())

status
Success    48252
Failed      1748
Name: count, dtype: int64

status
In Stock        922
Low Stock        76
Out of Stock      2
Name: count, dtype: int64


In [12]:
# Parse order_date once. Using format from the config cell.
orders["order_date"] = pd.to_datetime(orders["order_date"], format=DATE_FORMAT)
print(orders["order_date"].min(), "to", orders["order_date"].max())

# Data-quality flags
print("Duplicate order_id in orders        :", orders["order_id"].duplicated().sum())
print("Return order_ids not found in orders:", (~returns["order_id"].isin(orders["order_id"])).sum())
print("Null final_amount in orders         :", orders["final_amount"].isna().sum())
print("Order statuses:")
print(orders["status"].value_counts())

# Store every KPI result for the summary table later.
kpi = {}

2023-06-24 00:00:00 to 2025-06-23 00:00:00
Duplicate order_id in orders        : 0
Return order_ids not found in orders: 0
Null final_amount in orders         : 0
Order statuses:
status
Delivered     32499
Cancelled      5996
Shipped        5071
Returned       3933
Processing     2501
Name: count, dtype: int64


## KPI 1: Gross Merchandise Value (GMV)

- **Business question:** How big is the business before removing cancellations?
- **Formula:** Sum of all `final_amount` (all statuses)
- **Data:** `orders.final_amount`

In [13]:
# KPI 1: GMV
gmv = orders['final_amount'].sum()
kpi["gmv"] = gmv
print(f"Gross Merchandise Value (GMV) = Rs.{gmv:,.0f}")

Gross Merchandise Value (GMV) = Rs.3,147,385,545


## KPI 2: Net Revenue

- **Business question:** How much money did the platform actually earn?
- **Formula:** Sum of `final_amount` where `status = Delivered`
- **Data:** `orders.final_amount`, `orders.status`

In [14]:
# KPI 2: Net Revenue
delivered_orders = orders[orders["status"] == "Delivered"]
net_revenue = delivered_orders["final_amount"].sum()
kpi["net_revenue"] = net_revenue
print(f"Delivered orders = {len(delivered_orders):,}")
print(f"Net Revenue = Rs.{net_revenue:,.0f}")

Delivered orders = 32,499
Net Revenue = Rs.2,053,858,612


## KPI 3: Average Order Value (AOV)

- **Business question:** How much does a customer spend per delivered order?
- **Formula:** Net Revenue / Count of Delivered Orders


In [16]:
# KPI 3: AOV
delivered_count = len(delivered_orders)
if delivered_count > 0:
    aov = net_revenue / delivered_count
else:
    aov = None
kpi["aov"] = aov
print(f"Delivered orders = {delivered_count:,}")
print(f"AOV = Rs.{aov:,.0f}")

Delivered orders = 32,499
AOV = Rs.63,198


## KPI 4: Cancellation Rate

- **Business question:** Are we losing too many orders to cancellation?
- **Formula:** (Cancelled Orders / Total Orders) x 100
- **Target:** < 10%

In [19]:
# KPI 4: Cancellation Rate
total_orders = len(orders)
cancelled_orders = (orders["status"] == "Cancelled").sum()

if total_orders > 0:
  cancellation_rate = cancelled_orders / total_orders * 100
else:
  cancellation_rate = None

kpi["cancellation_rate"] = cancellation_rate

print(f"Cancelled = {cancelled_orders:,} of {total_orders:,} orders")
print(f"Cancellation Rate = {cancellation_rate:.2f}% (target < 10%)")

Cancelled = 5,996 of 50,000 orders
Cancellation Rate = 11.99% (target < 10%)


## KPI 5: Return Rate

- **Business question:** Is the return rate healthy?
- **Formula:** (Return records / Delivered Orders) x 100
- **Target:** < 8%
- **Data-quality effect:** If some return `order_id`s are missing from `orders.csv`, or one order has several return rows, numerator is slightly inflated vs a strict per-order rate. They are flagged not deleted

In [38]:
# KPI 5: Return Rate
return_records = len(returns)
if delivered_count > 0:
    return_rate = return_records / delivered_count * 100
else:
    return_rate = None
kpi["return_rate"] = return_rate

print(f"Return records = {return_records:,}")
print(f"Delivered orders = {delivered_count:,}")
print(f"Return Rate = {return_rate:.2f}% (target < 8%)")

# Flags
unmatched_returns = (~returns["order_id"].isin(orders["order_id"])).sum()
repeat_return_ids = returns["order_id"].duplicated().sum()
print("\nFlag - return rows with no matching order :", unmatched_returns)
print("Flag - extra return rows for same order_id:", repeat_return_ids)

Return records = 10,000
Delivered orders = 32,499
Return Rate = 30.77% (target < 8%)

Flag - return rows with no matching order : 0
Flag - extra return rows for same order_id: 0


## KPI 6: Customer Lifetime Value (CLV)

- **Business question:** Which customer segments are most valuable?
- **Formula:** Average `total_spent` per customer segment
- **Benchmark:** Premium segment should be about 3x Regular
- **Data:** `customers.segment`, `customers.total_spent` (pre-calculated lifetime spend)

In [28]:
# KPI 6: CLV
clv_by_segment = customers.groupby("segment")["total_spent"].mean().sort_values(ascending=False)
print("CLV by segment (average total_spent):")
for segment_name in clv_by_segment.index:
    print(f"  {segment_name:8s} Rs.{clv_by_segment[segment_name]:,.0f}")

# Premium vs Regular ratio (benchmark ~3x)
if "Premium" in clv_by_segment.index and "Regular" in clv_by_segment.index and clv_by_segment["Regular"] > 0:
    premium_to_regular = clv_by_segment["Premium"] / clv_by_segment["Regular"]
else:
    premium_to_regular = None
kpi["clv_ratio"] = premium_to_regular
print()
print(f"Premium / Regular CLV ratio = {premium_to_regular:.2f}x   (benchmark ~3x)")

CLV by segment (average total_spent):
  Premium  Rs.532,735
  Regular  Rs.361,871
  New      Rs.274,866
  Budget   Rs.186,179
  Inactive Rs.32,249

Premium / Regular CLV ratio = 1.47x   (benchmark ~3x)


## KPI 7: Month-over-Month (MoM) Growth

- **Business question:** Is the business growing month to month?
- **Formula:** ((This Month GMV - Last Month GMV) / Last Month GMV) x 100
- **Data:** `orders.order_date`, `orders.final_amount` (all statuses, same as GMV)
- **First month:** no previous month, so growth is left blank (NaN), not zero.

In [30]:
# KPI 7: MoM Growth
orders["order_month"] = orders["order_date"].dt.to_period("M")

# Monthly GMV, sorted oldest to newest
monthly_gmv = orders.groupby("order_month")["final_amount"].sum().sort_index()

# MoM growth using PRD formula
previous_month_gmv = monthly_gmv.shift(1)               # last month's GMV
mom_growth = (monthly_gmv - previous_month_gmv) / previous_month_gmv * 100

monthly_table = pd.DataFrame({
    "GMV (Rs.)": monthly_gmv.round(0),
    "MoM Growth (%)": mom_growth.round(2),
})
print(monthly_table)

kpi["mom_latest"] = mom_growth.iloc[-1]
kpi["mom_average"] = mom_growth.mean()                  # NaN of first month is skipped
print()
print(f"Latest month ({monthly_gmv.index[-1]}) MoM growth = {kpi['mom_latest']:.2f}%")
print(f"Average MoM growth (all months) = {kpi['mom_average']:.2f}%")

               GMV (Rs.)  MoM Growth (%)
order_month                             
2023-06       27384849.0             NaN
2023-07      128920509.0          370.77
2023-08      128122037.0           -0.62
2023-09      137462483.0            7.29
2023-10      132470874.0           -3.63
2023-11      137263359.0            3.62
2023-12      136923402.0           -0.25
2024-01      131450025.0           -4.00
2024-02      120945866.0           -7.99
2024-03      133670054.0           10.52
2024-04      131509731.0           -1.62
2024-05      136358367.0            3.69
2024-06      123740980.0           -9.25
2024-07      127028266.0            2.66
2024-08      131923460.0            3.85
2024-09      130525274.0           -1.06
2024-10      133518341.0            2.29
2024-11      126264142.0           -5.43
2024-12      133089460.0            5.41
2025-01      137929117.0            3.64
2025-02      114116741.0          -17.26
2025-03      136493234.0           19.61
2025-04      130

## KPI 8: Top Category Revenue Share

- **Business question:** Is revenue too concentrated in one category?
- **Formula:** Category Revenue / Total Revenue x 100
- **Risk threshold:** Concentration risk if one category > 60%
- **Data:** `order_items.category`, `order_items.total_price`

In [40]:
# KPI 8: Top Category Revenue
category_revenue = order_items.groupby("category")["total_price"].sum().sort_values(ascending=False)
total_category_revenue = category_revenue.sum()

if total_category_revenue > 0:
    category_share = category_revenue / total_category_revenue * 100
else:
    category_share = None

top_category = category_revenue.index[0]
top_category_share = category_share.iloc[0]
kpi["top_category"] = top_category
kpi["top_category_share"] = top_category_share

print(category_share.round(2))
print()
print(f"Top category = {top_category} | Revenue Rs.{category_revenue.iloc[0]:,.0f} | Share {top_category_share:.2f}%")

category
Electronics         57.41
Sports & Fitness    17.38
Home & Kitchen       8.54
Fashion              5.18
Automotive           4.32
Office Supplies      3.25
Toys & Baby          1.63
Beauty & Health      1.15
Grocery              0.65
Books                0.49
Name: total_price, dtype: float64

Top category = Electronics | Revenue Rs.1,642,422,813 | Share 57.41%


## KPI 9: Payment Failure Rate

- **Business question:** Are failed payments causing revenue leakage?
- **Formula:** (Failed Payments / Total Payments) x 100
- **Target:** < 2%
- **Data:** `payments` status column

In [33]:
total_payments = len(payments)
failed_payments = (payments[PAYMENT_STATUS_COL] == PAYMENT_FAILED_VALUE).sum()

if total_payments > 0:
    payment_failure_rate = failed_payments / total_payments * 100
else:
    payment_failure_rate = None
kpi["payment_failure_rate"] = payment_failure_rate

print(f"Failed = {failed_payments:,} of {total_payments:,} payments")
print(f"Payment Failure Rate = {payment_failure_rate:.2f}% (target < 2%)")

Failed = 1,748 of 50,000 payments
Payment Failure Rate = 3.50% (target < 2%)


## KPI 10: Inventory Fill Rate

- **Business question:** Are products available to sell?
- **Formula:** (In Stock SKUs / Total SKUs) x 100
- **Target:** > 95%
- **Data:** `inventory` status column

In [34]:
# KPI 10: Inventory Fill Rate
total_skus = len(inventory)
in_stock_skus = (inventory[INVENTORY_STATUS_COL] == INVENTORY_IN_STOCK_VALUE).sum()

if total_skus > 0:
    inventory_fill_rate = in_stock_skus / total_skus * 100
else:
    inventory_fill_rate = None
kpi["inventory_fill_rate"] = inventory_fill_rate

print(f"In Stock = {in_stock_skus:,} of {total_skus:,} SKUs")
print(f"Inventory Fill Rate = {inventory_fill_rate:.2f}% (target > 95%)")

In Stock = 922 of 1,000 SKUs
Inventory Fill Rate = 92.20% (target > 95%)


## Consolidated KPI Summary

In [39]:
def_status = {}   # PASS / REVIEW flags against benchmark

def_status["cancel"] = "PASS" if kpi["cancellation_rate"] < 10 else "REVIEW"
def_status["return"] = "PASS" if kpi["return_rate"] < 8 else "REVIEW"
def_status["clv"]    = "PASS" if (kpi["clv_ratio"] is not None and kpi["clv_ratio"] >= 3) else "REVIEW"
def_status["cat"]    = "PASS" if kpi["top_category_share"] <= 60 else "REVIEW"
def_status["pay"]    = "PASS" if kpi["payment_failure_rate"] < 2 else "REVIEW"
def_status["fill"]   = "PASS" if kpi["inventory_fill_rate"] > 95 else "REVIEW"

summary = pd.DataFrame({
    "KPI": ["1. GMV", "2. Net Revenue", "3. AOV", "4. Cancellation Rate", "5. Return Rate",
            "6. CLV (Premium / Regular)", "7. MoM Growth (latest month)",
            "8. Top Category Share", "9. Payment Failure Rate", "10. Inventory Fill Rate"],
    "Value": [f"Rs.{kpi['gmv']:,.0f}", f"Rs.{kpi['net_revenue']:,.0f}", f"Rs.{kpi['aov']:,.0f}",
              f"{kpi['cancellation_rate']:.2f}%", f"{kpi['return_rate']:.2f}%",
              f"{kpi['clv_ratio']:.2f}x", f"{kpi['mom_latest']:.2f}%",
              f"{kpi['top_category']}: {kpi['top_category_share']:.2f}%",
              f"{kpi['payment_failure_rate']:.2f}%", f"{kpi['inventory_fill_rate']:.2f}%"],
    "Benchmark": ["-", "-", "-", "< 10%", "< 8%", "Premium ~3x Regular",
                      "-", "Risk if > 60%", "< 2%", "> 95%"],
    "Status": ["-", "-", "-", def_status["cancel"], def_status["return"], def_status["clv"],
               "-", def_status["cat"], def_status["pay"], def_status["fill"]],
})
summary

,KPI,Value,Benchmark,Status
0,1. GMV,"Rs.3,147,385,545",-,-
1,2. Net Revenue,"Rs.2,053,858,612",-,-
2,3. AOV,"Rs.63,198",-,-
3,4. Cancellation Rate,11.99%,< 10%,REVIEW
4,5. Return Rate,30.77%,< 8%,REVIEW
5,6. CLV (Premium / Regular),1.47x,Premium ~3x Regular,REVIEW
6,7. MoM Growth (latest month),-23.74%,-,-
7,8. Top Category Share,Electronics: 57.41%,Risk if > 60%,PASS
8,9. Payment Failure Rate,3.50%,< 2%,REVIEW
9,10. Inventory Fill Rate,92.20%,> 95%,REVIEW


## Business Interpretation vs Benchmarks

In [36]:
print(f"GMV Rs.{kpi['gmv']:,.0f} vs Net Revenue Rs.{kpi['net_revenue']:,.0f}: "
      f"{kpi['net_revenue'] / kpi['gmv'] * 100:.1f}% of order value ended as delivered revenue.")

print(f"AOV Rs.{kpi['aov']:,.0f} per delivered order.")

print(f"Cancellation rate {kpi['cancellation_rate']:.2f}% -> {def_status['cancel']} (target < 10%).")
print(f"Return rate {kpi['return_rate']:.2f}% -> {def_status['return']} (target < 8%).")
print(f"Premium CLV is {kpi['clv_ratio']:.2f}x Regular -> {def_status['clv']} (benchmark ~3x).")

print(f"Latest MoM growth {kpi['mom_latest']:.2f}%; average {kpi['mom_average']:.2f}%.")
print(f"{kpi['top_category']} leads with {kpi['top_category_share']:.2f}% of category revenue -> {def_status['cat']} (risk if > 60%).")

print(f"Payment failure {kpi['payment_failure_rate']:.2f}% -> {def_status['pay']} (target < 2%).")
print(f"Inventory fill rate {kpi['inventory_fill_rate']:.2f}% -> {def_status['fill']} (target > 95%).")

GMV Rs.3,147,385,545 vs Net Revenue Rs.2,053,858,612: 65.3% of order value ended as delivered revenue.
AOV Rs.63,198 per delivered order.
Cancellation rate 11.99% -> REVIEW (target < 10%).
Return rate 30.77% -> REVIEW (target < 8%).
Premium CLV is 1.47x Regular -> REVIEW (benchmark ~3x).
Latest MoM growth -23.74%; average 14.92%.
Electronics leads with 57.41% of category revenue -> PASS (risk if > 60%).
Payment failure 3.50% -> REVIEW (target < 2%).
Inventory fill rate 92.20% -> REVIEW (target > 95%).


## Final Validation Check

In [37]:
# Final Validation check
checks = {}
checks["Order status counts add up to total orders"] = orders["status"].value_counts().sum() == len(orders)
checks["Net Revenue <= GMV"] = kpi["net_revenue"] <= kpi["gmv"]
checks["No null final_amount (nulls are skipped by sum)"] = orders["final_amount"].isna().sum() == 0
checks["All 4 rates between 0 and 100"] = all(0 <= r <= 100 for r in [
    kpi["cancellation_rate"], kpi["return_rate"], kpi["payment_failure_rate"], kpi["inventory_fill_rate"]])
checks["Category shares add up to 100"] = abs(category_share.sum() - 100) < 0.001
checks["Monthly GMV adds up to total GMV"] = abs(monthly_gmv.sum() - kpi["gmv"]) < 1
checks["First month MoM is blank (no previous month)"] = pd.isna(mom_growth.iloc[0])
checks["AOV x Delivered orders = Net Revenue"] = abs(kpi["aov"] * delivered_count - kpi["net_revenue"]) < 1

for name in checks:
    print("PASS" if checks[name] else "FAIL", "-", name)

print()
print("Duplicate order_id rows       :", orders["order_id"].duplicated().sum())
print("Return rows without order     :", unmatched_returns)
print("Extra return rows, same order :", repeat_return_ids)

PASS - Order status counts add up to total orders
PASS - Net Revenue <= GMV
PASS - No null final_amount (nulls are skipped by sum)
PASS - All 4 rates between 0 and 100
PASS - Category shares add up to 100
PASS - Monthly GMV adds up to total GMV
PASS - First month MoM is blank (no previous month)
PASS - AOV x Delivered orders = Net Revenue

Duplicate order_id rows       : 0
Return rows without order     : 0
Extra return rows, same order : 0
